# Phase 3.2: Mechanism Ablations - Hidden State

Analyze how hidden-state mechanisms affect c-GC and c-GC* recovery across depths.

## Scenarios
- **latent_confounder**: Latent common driver (unmeasured confounder)
- **hidden_nodes**: Some nodes completely hidden from observation
- **omitted_lag_order**: Misspecified lag order (true order=3, assume order=1)

For each scenario:
- Run c-GC and c-GC* across depths [1,2,3,4,5,6]
- Compute recovery metrics (accuracy, precision, recall, FPR)
- Track D_p trajectories and instability signatures
- Export results.csv, summary.json, figures/

## Setup: Imports and Configuration

In [1]:
from __future__ import annotations

import sys
import json
import logging
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Find project root
CAUSALISED_GC_RELATIVE_PATH = Path('src/markovianity_diagnostic/core/causalised-GC.py')
PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / CAUSALISED_GC_RELATIVE_PATH).exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f'Could not find {CAUSALISED_GC_RELATIVE_PATH} from {Path.cwd().resolve()}'
    )

# Add src to path for imports
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from markovianity_diagnostic.experiments.simulations import (
    scenario_latent_common_driver,
    scenario_hidden_nodes,
    scenario_omitted_lag_order,
)
from markovianity_diagnostic.experiments.adapters import METHODS
from markovianity_diagnostic.experiments.graph_metrics import summarize_run

logger.info(f'Project root: {PROJECT_ROOT}')
print(f'Project root: {PROJECT_ROOT}')

2026-07-05 22:03:08,749 - __main__ - INFO - Project root: /Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics


Project root: /Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics


In [2]:
# Configuration
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'simulations' / 'mechanism_ablations' / 'hidden_state'
FIGURES_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Experimental parameters
SEED = 42
REPEATS = 10
T = 2000
D = 10
P_VALUES = [1, 2, 3, 4, 5, 6, 7]
METHODS_TO_TEST = ['gcstar_cgc', 'gcstar_fcgc']

# Scenario-specific parameters
SCENARIOS = [
    ('latent_confounder', {
        'T': T,
        'd': D,
        'conf_strength': 0.35,
        'latent_ar': 0.80,
        'noise_scale': 1.0,
    }),
    ('hidden_nodes', {
        'T': T,
        'd_total': 16,
        'd_observed': D,
        'noise_scale': 1.0,
    }),
    ('omitted_lag_order', {
        'T': T,
        'd': D,
        'noise_scale': 1.0,
    }),
]

logger.info(f'Output directory: {OUTPUT_DIR}')
logger.info(f'Repeats: {REPEATS}')
logger.info(f'P-values: {P_VALUES}')
logger.info(f'Methods: {METHODS_TO_TEST}')

print(f'Output directory: {OUTPUT_DIR}')
print(f'Repeats: {REPEATS}')
print(f'P-values: {P_VALUES}')
print(f'Methods: {METHODS_TO_TEST}')

2026-07-05 22:03:08,773 - __main__ - INFO - Output directory: /Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics/outputs/simulations/mechanism_ablations/hidden_state
2026-07-05 22:03:08,774 - __main__ - INFO - Repeats: 10
2026-07-05 22:03:08,778 - __main__ - INFO - P-values: [1, 2, 3, 4, 5, 6, 7]
2026-07-05 22:03:08,783 - __main__ - INFO - Methods: ['gcstar_cgc', 'gcstar_fcgc']


Output directory: /Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics/outputs/simulations/mechanism_ablations/hidden_state
Repeats: 10
P-values: [1, 2, 3, 4, 5, 6, 7]
Methods: ['gcstar_cgc', 'gcstar_fcgc']


In [3]:
# Check for cached outputs
expected_outputs = {
    'results.csv': OUTPUT_DIR / 'results.csv',
    'manifest.json': OUTPUT_DIR / 'manifest.json',
    'figures/dp_trajectories.png': FIGURES_DIR / 'dp_trajectories.png',
}

outputs_exist = all(fpath.exists() for fpath in expected_outputs.values())

if outputs_exist:
    logger.info("Outputs already exist - loading cached results")
    print("✅ Outputs already exist - loading cached results")
    print(f"Output directory: {OUTPUT_DIR}")
    for name, path in expected_outputs.items():
        if path.exists():
            size_mb = path.stat().st_size / (1024 * 1024)
            logger.info(f"  ✓ {name} ({size_mb:.2f} MB)")
            print(f"  ✓ {name} ({size_mb:.2f} MB)")
else:
    logger.info("No existing outputs - will run computation")
    print("⚠️ No existing outputs - will run computation")
    print(f"Output directory: {OUTPUT_DIR}")

2026-07-05 22:03:08,803 - __main__ - INFO - No existing outputs - will run computation


⚠️ No existing outputs - will run computation
Output directory: /Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics/outputs/simulations/mechanism_ablations/hidden_state


In [4]:
if not outputs_exist:
    logger.info("Starting mechanism ablation experiments (hidden state)...")
    start_experiments = time.time()
    
    scenario_functions = {
        'latent_confounder': scenario_latent_common_driver,
        'hidden_nodes': scenario_hidden_nodes,
        'omitted_lag_order': scenario_omitted_lag_order,
    }
    all_results = {}
    
    for scenario_idx, (scenario_name, scenario_kwargs) in enumerate(SCENARIOS, 1):
        logger.info(f"\n[{scenario_idx}/{len(SCENARIOS)}] Scenario: {scenario_name}")
        print(f"\nScenario: {scenario_name}")
        scenario_fn = scenario_functions[scenario_name]
        all_results[scenario_name] = {}
        
        scenario_start = time.time()
        
        for method_idx, method_name in enumerate(METHODS_TO_TEST, 1):
            logger.info(f"  [{method_idx}/{len(METHODS_TO_TEST)}] Method: {method_name}")
            
            if method_name not in METHODS:
                logger.warning(f"    Warning: {method_name} not found")
                print(f'    Warning: {method_name} not found')
                continue
            
            analyze_fn = METHODS[method_name]
            method_results = []
            
            method_start = time.time()
            for repeat_idx in range(REPEATS):
                logger.info(f"      Repeat [{repeat_idx+1}/{REPEATS}]...")
                
                sample = scenario_fn(**scenario_kwargs, seed=SEED + repeat_idx)
                X = sample.X
                truth_compact = sample.ground_truth_compact
                
                adjacencies_by_p = analyze_fn(X, P_VALUES)
                run_summary = summarize_run(adjacencies_by_p, truth_compact)
                
                result = {
                    'repeat': repeat_idx,
                    'metadata': sample.metadata,
                    'summary': run_summary,
                }
                method_results.append(result)
            
            method_elapsed = time.time() - method_start
            logger.info(f"    ✓ {method_name} completed (elapsed: {method_elapsed:.2f}s)")
            print(f'    ✓ {method_name} completed')
            all_results[scenario_name][method_name] = method_results
        
        scenario_elapsed = time.time() - scenario_start
        logger.info(f"  Scenario {scenario_name} completed (elapsed: {scenario_elapsed:.2f}s)")
    
    total_elapsed = time.time() - start_experiments
    logger.info(f"\nAll experiments completed (total: {total_elapsed:.2f}s)")
    print('✓ All experiments completed')
else:
    logger.info("Skipping experiments (loading from cache)")
    print("Skipping experiments (loading from cache)")

2026-07-05 22:03:08,827 - __main__ - INFO - Starting mechanism ablation experiments (hidden state)...
2026-07-05 22:03:08,828 - __main__ - INFO - 
[1/3] Scenario: latent_confounder
2026-07-05 22:03:08,832 - __main__ - INFO -   [1/2] Method: gcstar_cgc
2026-07-05 22:03:08,833 - __main__ - INFO -       Repeat [1/10]...



Scenario: latent_confounder
    ✓ gcstar_cgc completed
    ✓ gcstar_fcgc completed

Scenario: hidden_nodes
    ✓ gcstar_cgc completed
    ✓ gcstar_fcgc completed

Scenario: omitted_lag_order


/Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics/.venv/lib/python3.12/site-packages/numpy/linalg/_linalg.py:2803: RuntimeWarning: overflow encountered in multiply
  s = (x.conj() * x).real
/Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics/.venv/lib/python3.12/site-packages/numpy/linalg/_linalg.py:2804: RuntimeWarning: overflow encountered in reduce
  return sqrt(add.reduce(s, axis=axis, keepdims=keepdims))
/Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics/src/markovianity_diagnostic/core/fast_causalised_GC.py:148: RuntimeWarning: overflow encountered in multiply
  cross = np.fft.ifft(fz * np.conj(fz[j]), axis=1).real  # (rows, T')
/Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics/.venv/lib/python3.12/site-packages/numpy/f

    ✓ gcstar_cgc completed


/Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics/src/markovianity_diagnostic/core/fast_causalised_GC.py:180: RuntimeWarning: overflow encountered in matmul
  sigma = (zc @ zc.T) / t


    ✓ gcstar_fcgc completed
✓ All experiments completed


In [5]:
# Load cached results if they exist
if outputs_exist:
    summary_csv_path = OUTPUT_DIR / 'results.csv'
    summary_df = pd.read_csv(summary_csv_path)
    print(f"Loaded cached results: {len(summary_df)} rows")
    print(summary_df)

In [6]:
if not outputs_exist:
    # Summary CSV
    summary_rows = []
    for scenario_name, method_results in all_results.items():
        for method_name, results in method_results.items():
            T_obs_values = [r['summary'].get('T_obs', 0.0) for r in results]
            row = {
                'scenario': scenario_name,
                'method': method_name,
                'repeats': len(results),
                'mean_T_obs': float(np.mean(T_obs_values)) if T_obs_values else 0.0,
            }
            summary_rows.append(row)
    summary_df = pd.DataFrame(summary_rows)
    summary_csv_path = OUTPUT_DIR / 'results.csv'
    summary_df.to_csv(summary_csv_path, index=False)
    print(f'Exported: {summary_csv_path}')
else:
    print("Skipping export (loading from cache)")

Exported: /Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics/outputs/simulations/mechanism_ablations/hidden_state/results.csv


In [7]:
# Figures (always run for visualization)
if not outputs_exist and 'all_results' in locals():
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for scenario_idx, (scenario_name, method_results) in enumerate(all_results.items()):
        ax = axes[scenario_idx]
        for method_name, results in method_results.items():
            d_means = {}
            for p_val in P_VALUES[1:]:
                d_values = []
                for result in results:
                    d_val = result['summary'].get('D_p', {}).get(p_val)
                    if d_val is not None:
                        d_values.append(float(d_val))
                d_means[p_val] = np.mean(d_values) if d_values else 0.0
            p_vals_sorted = sorted(d_means.keys())
            d_vals_sorted = [d_means[p] for p in p_vals_sorted]
            style = '-o' if method_name == 'gcstar_cgc' else '--s'
            ax.plot(p_vals_sorted, d_vals_sorted, style, label=method_name, linewidth=2)
        ax.set_xlabel('Conditioning Depth p')
        ax.set_ylabel('Mean D_p')
        ax.set_title(scenario_name)
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    dp_traj_path = FIGURES_DIR / 'dp_trajectories.png'
    plt.savefig(dp_traj_path, dpi=100, bbox_inches='tight')
    print(f'Exported: {dp_traj_path}')
    plt.close()
else:
    print("Skipping figure creation (using cached outputs)")

Exported: /Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics/outputs/simulations/mechanism_ablations/hidden_state/figures/dp_trajectories.png


In [8]:
# Display the summary created above or loaded from cache.
if 'summary_df' not in locals():
    raise RuntimeError('Summary results are unavailable; run the notebook from the first cell.')
print(summary_df)

            scenario       method  repeats  mean_T_obs
0  latent_confounder   gcstar_cgc       10    0.033333
1  latent_confounder  gcstar_fcgc       10    0.018889
2       hidden_nodes   gcstar_cgc       10    0.030000
3       hidden_nodes  gcstar_fcgc       10    0.026667
4  omitted_lag_order   gcstar_cgc       10    0.024444
5  omitted_lag_order  gcstar_fcgc       10    0.025556


In [9]:
if not outputs_exist:
    # Manifest
    manifest = {
        'created_at': datetime.now(timezone.utc).isoformat(),
        'analysis': 'mechanism_ablations_hidden_state',
        'scenarios': list(all_results.keys()),
        'methods': METHODS_TO_TEST,
        'p_values': P_VALUES,
        'repeats': REPEATS,
    }
    manifest_path = OUTPUT_DIR / 'manifest.json'
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)
    print(f'Exported: {manifest_path}')
    print('✓ All outputs verified successfully')
else:
    print("Skipping manifest creation (loading from cache)")

Exported: /Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics/outputs/simulations/mechanism_ablations/hidden_state/manifest.json
✓ All outputs verified successfully


## Manifest and Verification

In [10]:
logger.info("Creating manifest...")

# Manifest
manifest = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'analysis': 'mechanism_ablations_hidden_state',
    'scenarios': list(all_results.keys()) if 'all_results' in locals() else [s[0] for s in SCENARIOS],
    'methods': METHODS_TO_TEST,
    'p_values': P_VALUES,
    'repeats': REPEATS,
    'total_experiments': REPEATS * len(METHODS_TO_TEST) * len(SCENARIOS),
}
manifest_path = OUTPUT_DIR / 'manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
logger.info(f'Exported: {manifest_path}')
logger.info('✓ All outputs verified successfully')
print(f'Exported: {manifest_path}')
print('✓ All outputs verified successfully')

Exported: /Users/sadiqadedayo/Documents/projects/Detecting Latent confounders with Markovianity/hidden-confounding-diagnostics/outputs/simulations/mechanism_ablations/hidden_state/manifest.json
✓ All outputs verified successfully
